# SPECTRA Baseline for Topic Modeling Analysis

This notebook provides a baseline implementation using [SPECTRA (Supervised Pathway DEConvolution of InTerpretable Gene ProgRAms)](https://github.com/dpeerlab/spectra) for single-cell topic modeling analysis.

SPECTRA takes in a single cell gene expression matrix, cell type annotations, and gene sets for cellular processes to fit interpretable factors (topics) from the data, extracting topic-gene matrices for comparison with scE2TM.

## 📋 Prerequisites

Before running this baseline, please ensure you have:

1. **Followed the main installation guide** in the root directory's README to set up the conda environment

2. **Installed additional dependencies** specific to SPECTRA:

```bash
# Activate the scE2TM environment first
conda activate scE2TM_env

# Install SPECTRA
pip install scSpectra

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from pathlib import Path
import Spectra as spc
from Spectra import Spectra_util as spc_tl
from Spectra import default_gene_sets
import warnings
warnings.filterwarnings('ignore')

# ========== Path Configuration ==========
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_ROOT / 'data'

DATASET_NAME = 'Wang'

print("="*60)
print("Baseline: SPECTRA Topic Analysis")
print(f"Dataset: {DATASET_NAME}")
print("="*60)

# ========== Load data ==========
print("\n1. Loading data...")

data_path = DATA_DIR / f'{DATASET_NAME}_HIGHPRE.csv'
label_path = DATA_DIR / f'{DATASET_NAME}_cell_anno.csv'

# Read expression data
expression_df = pd.read_csv(data_path, index_col=0)
adata = sc.AnnData(expression_df)
print(f"   Expression data shape: {adata.shape}")

# Load labels if available
if label_path.exists():
    label_df = pd.read_csv(label_path, index_col=0).iloc[:, 0]
    adata.obs['cell_type'] = label_df.values
    print(f"   Labels loaded: {len(adata.obs['cell_type'].unique())} cell types")

# ========== Load gene sets ==========
print("\n2. Loading gene sets...")

annotations = spc.default_gene_sets.load()

annotations = spc_tl.check_gene_set_dictionary(
    adata,
    annotations,
    obs_key='cell_type',
    use_cell_types=False,
    global_key='global'
)

annotations_dict = {"global": annotations}
print(f"   Gene sets loaded")

# ========== Run SPECTRA ==========
print("\n3. Running SPECTRA...")

model = spc.est_spectra(
    adata=adata,
    L=10,  # Number of topics/factors
    gene_set_dictionary=annotations_dict,
    use_highly_variable=False,
    cell_type_key="cell_type",
    use_weights=True,
    lam=0.1,
    delta=0.001,
    kappa=None,
    rho=0.001,
    use_cell_types=False,
    n_top_vals=50,
    label_factors=True,
    overlap_threshold=0.2,
    clean_gs=True,
    min_gs_num=3,
    num_epochs=10000
)

print(f"\n   SPECTRA completed!")

# ========== Extract topic-gene matrix ==========
print("\n4. Extracting topic-gene matrix...")

# factors: genes x topics matrix (topic loadings)
factors = adata.uns['SPECTRA_factors']
print(f"   Topic-gene matrix shape: {factors.shape} (genes x topics)")

# ========== Extract top genes per topic ==========
print("\n5. Top 20 genes per topic:")

gene_names = adata.var_names.tolist()
k = factors.shape[1]  # Number of topics

topic_top_genes = {}
for topic_idx in range(k):
    scores = factors[:, topic_idx]
    top_indices = np.argsort(scores)[-20:][::-1]
    top_genes_list = [gene_names[i] for i in top_indices]
    topic_top_genes[topic_idx] = top_genes_list

# Display top genes per topic
for i in range(k):
    print(f"\n   Topic {i}:")
    for j, gene in enumerate(topic_top_genes[i][:10]):  # Show first 10
        print(f"      {j+1}. {gene}")

# ========== Summary ==========
print("\n" + "="*60)
print("Summary")
print("="*60)
print(f"Dataset: {DATASET_NAME}")
print(f"Number of topics (K): {k}")
print(f"Topic-gene matrix shape: {factors.shape} (genes x topics)")

print("\n" + "="*60)
print("SPECTRA baseline completed!")
print("="*60)